# NB9 — RF-DETR + FashionCLIP Core-7 Detection V1 — Colab smoke test

Notebook này kiểm tra **runtime + chất lượng wiring** của detection branch trên ảnh thật:

```text
image
→ RF-DETR garment boxes
→ crops
→ FashionCLIP 512-d L2 embeddings
→ zero-shot Core-7 category
→ scorer tensor handoff
→ frozen V5 scorer smoke test (nếu có 3–8 garments)
```

Ngoài canonical output, notebook còn in **diagnostic RF-DETR label → Core-7 vs FashionCLIP → Core-7** để xem zero-shot classifier có đang đoán lại sai một category mà detector đã nhận ra rõ hay không. Diagnostic này **không thay đổi** implementation canonical.


In [ ]:
from pathlib import Path
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
BRANCH = "feat/detection-rfdetr-fashionclip-core7"
REPO_ROOT = Path("/content/opisoverated")

if not REPO_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )

os.chdir(REPO_ROOT)

# Important when the same Colab runtime previously imported src.* from another branch.
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        sys.modules.pop(module_name, None)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements-detection.txt")],
    check=True,
)

HEAD = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True
).strip()
CURRENT_BRANCH = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "branch", "--show-current"], text=True
).strip()

print("Repo   :", REPO_ROOT)
print("Branch :", CURRENT_BRANCH)
print("HEAD   :", HEAD)


In [ ]:
# Fail fast on lightweight contract tests before downloading model weights.
test_run = subprocess.run(
    [
        sys.executable, "-m", "unittest", "discover",
        "-s", "tests", "-p", "test_detection_core7.py", "-v",
    ],
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
if test_run.returncode != 0:
    raise RuntimeError(f"Detection contract tests failed: {test_run.returncode}")
print("DETECTION CONTRACT TESTS: PASS")


In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "Select Runtime → Change runtime type → T4 GPU (or another CUDA GPU), then Run all again."
    )
print("GPU:", torch.cuda.get_device_name(0))


## 1. Chọn ảnh

Mặc định notebook dùng ảnh sample đã được commit ở `tests/animage.jpg`, nên có thể **Run all** mà không cần tạo đường dẫn thủ công.

Muốn thử ảnh riêng, đặt `UPLOAD_CUSTOM_IMAGE = True`; Colab sẽ mở hộp upload.


In [ ]:
UPLOAD_CUSTOM_IMAGE = False
REPO_SAMPLE = REPO_ROOT / "tests" / "animage.jpg"

if UPLOAD_CUSTOM_IMAGE:
    try:
        from google.colab import files
    except ImportError as error:
        raise RuntimeError("Custom upload is supported in Google Colab.") from error

    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No image uploaded.")
    uploaded_name = next(iter(uploaded))
    IMAGE_PATH = (Path.cwd() / uploaded_name).resolve()
else:
    IMAGE_PATH = REPO_SAMPLE

if not IMAGE_PATH.is_file():
    raise FileNotFoundError(
        f"Image not found: {IMAGE_PATH}. "
        "The committed sample should be tests/animage.jpg; otherwise set UPLOAD_CUSTOM_IMAGE=True."
    )

print("IMAGE_PATH:", IMAGE_PATH)


## 2. Run RF-DETR + FashionCLIP


In [ ]:
from src.detection import DetectionPipeline, load_detection_config

CONFIG_PATH = REPO_ROOT / "configs" / "detection_rfdetr_fashionclip_core7_v1.json"
config = load_detection_config(CONFIG_PATH)

pipeline = DetectionPipeline(config, device="cuda")
result, image = pipeline.run(IMAGE_PATH)

print("image size          :", image.size)
print("accepted garments   :", len(result.garments))
print("rejected detections :", len(result.rejected_detections))
print("RF-DETR runtime ms  :", result.detector_runtime_ms)

for index, garment in enumerate(result.garments):
    confidence = garment.candidate.detector_confidence
    confidence_text = "n/a" if confidence is None else f"{confidence:.3f}"
    print(
        f"[{index}] {garment.candidate.detector_label!r} "
        f"det_conf={confidence_text} → {garment.category.coarse_category} "
        f"sim={garment.category.similarity:+.4f} "
        f"margin={garment.category.margin:.4f}"
    )

if result.rejected_detections:
    print("\nRejected examples:")
    for row in result.rejected_detections[:20]:
        print(row)


## 3. Visualize accepted boxes

Đây là check quan trọng nhất cho **localization**: RF-DETR có lấy đúng garment/accessory cần cho scorer hay không.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(image)

for garment in result.garments:
    x0, y0, x1, y1 = garment.crop_box_xyxy
    rect = patches.Rectangle(
        (x0, y0),
        x1 - x0,
        y1 - y0,
        fill=False,
        linewidth=2,
    )
    ax.add_patch(rect)

    confidence = garment.candidate.detector_confidence
    confidence_text = "n/a" if confidence is None else f"{confidence:.2f}"
    label = (
        f"{garment.candidate.detector_label} → {garment.category.coarse_category}\n"
        f"det={confidence_text}, sim={garment.category.similarity:.3f}, "
        f"margin={garment.category.margin:.3f}"
    )
    ax.text(x0, max(0, y0 - 4), label, fontsize=8)

ax.axis("off")
plt.tight_layout()
plt.show()


## 4. Diagnostic: RF-DETR label vs FashionCLIP Core-7

RF-DETR label hiện chỉ là **candidate filter**, còn canonical Core-7 prediction đến từ FashionCLIP zero-shot.

Bảng dưới **không đổi prediction**. Nó chỉ kiểm tra một câu hỏi thực nghiệm:

> Nếu RF-DETR đã nói `shoe`, FashionCLIP có thực sự nên được phép đổi nó thành `TOP/BOTTOM/...` không?

Nếu hai nguồn thường bất đồng trên ảnh thật, nên A/B `detector-label mapping` vs `FashionCLIP zero-shot` trước khi freeze detection architecture.


In [ ]:
DETECTOR_LABEL_TO_CORE7 = {
    "shirt, blouse": "TOP",
    "top, t-shirt, sweatshirt": "TOP",
    "sweater": "TOP",
    "cardigan": "OUTERWEAR",
    "jacket": "OUTERWEAR",
    "vest": "OUTERWEAR",
    "pants": "BOTTOM",
    "shorts": "BOTTOM",
    "skirt": "BOTTOM",
    "coat": "OUTERWEAR",
    "dress": "DRESS",
    "jumpsuit": "DRESS",
    "cape": "OUTERWEAR",
    "hat": "HAT",
    "shoe": "SHOES",
    "bag, wallet": "BAG",
}

agreement_count = 0
for index, garment in enumerate(result.garments):
    detector_core7 = DETECTOR_LABEL_TO_CORE7.get(garment.candidate.detector_label)
    fashionclip_core7 = garment.category.coarse_category
    agrees = detector_core7 == fashionclip_core7
    agreement_count += int(agrees)

    embedding = garment.embedding.float()
    embedding_norm = float(torch.linalg.vector_norm(embedding).item())
    if abs(embedding_norm - 1.0) > 1e-4:
        raise ValueError(
            f"Garment {index} embedding is not L2-normalized: norm={embedding_norm}"
        )

    print(
        f"[{index}] detector={garment.candidate.detector_label!r:28s} "
        f"detector→Core7={str(detector_core7):10s} "
        f"FashionCLIP→Core7={fashionclip_core7:10s} "
        f"agree={str(agrees):5s} "
        f"sim={garment.category.similarity:+.4f} "
        f"margin={garment.category.margin:.4f} "
        f"||emb||={embedding_norm:.6f}"
    )

if result.garments:
    print(
        "\nDetector-label / FashionCLIP category agreement:",
        f"{agreement_count}/{len(result.garments)} "
        f"({agreement_count / len(result.garments):.1%})",
    )


## 5. Save crops + inspect them


In [ ]:
from src.detection.pipeline import save_detection_result

OUTPUT_ROOT = REPO_ROOT / "outputs" / "detection_v1"
run_dir = OUTPUT_ROOT / IMAGE_PATH.stem

saved = save_detection_result(
    result,
    image,
    run_dir,
    scorer_min_items=config.scorer_min_items,
    scorer_max_items=config.scorer_max_items,
)
print(saved)


In [ ]:
from PIL import Image

crop_paths = sorted((run_dir / "crops").glob("*.jpg"))
for crop_path in crop_paths:
    plt.figure(figsize=(3, 3))
    plt.imshow(Image.open(crop_path))
    plt.title(crop_path.stem)
    plt.axis("off")
    plt.show()


## 6. Inspect metadata + scorer handoff


In [ ]:
import json

metadata_path = run_dir / "detection_result.json"
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
print(json.dumps(metadata, ensure_ascii=False, indent=2)[:12000])

scorer_path = saved["scorer_inputs_path"]
if scorer_path is None:
    print("\nSCORER HANDOFF SKIPPED:", saved["scorer_handoff_error"])
else:
    scorer_inputs = torch.load(scorer_path, map_location="cpu")
    print("\nscorer_inputs.pt")
    for key, value in scorer_inputs.items():
        print(key, tuple(value.shape), value.dtype)

    assert scorer_inputs["item_embeddings"].shape[0] == 1
    assert scorer_inputs["item_embeddings"].shape[-1] == 512
    assert scorer_inputs["coarse_category_ids"].shape == scorer_inputs["item_mask"].shape
    norms = torch.linalg.vector_norm(scorer_inputs["item_embeddings"].float(), dim=-1)
    print("embedding norms:", norms)
    if not torch.allclose(norms, torch.ones_like(norms), atol=1e-4, rtol=0.0):
        raise ValueError("scorer_inputs contains non-L2-normalized item embeddings")

    print("SCORER HANDOFF TENSORS: PASS")


## 7. Frozen V5 scorer smoke test

Chỉ chạy nếu detection tạo được số item hợp lệ theo scorer contract (`3–8`).


In [ ]:
if scorer_path is not None:
    from src.scorer.checkpoint import load_checkpoint
    from src.scorer.model import TypeAwarePairwiseScorer

    BEST_PATH = (
        REPO_ROOT
        / "artifacts"
        / "checkpoints"
        / "type_aware_pairwise_v1"
        / "final_val_auc_v5_seed42"
        / "best.pt"
    )
    if not BEST_PATH.is_file():
        raise FileNotFoundError(f"Missing frozen V5 checkpoint: {BEST_PATH}")

    payload = load_checkpoint(BEST_PATH, map_location="cpu")
    scorer = TypeAwarePairwiseScorer.from_config(payload["config"])
    scorer.load_state_dict(payload["model_state_dict"])
    scorer.to("cuda").eval()

    batch = {key: value.to("cuda") for key, value in scorer_inputs.items()}
    with torch.inference_mode():
        scorer_output = scorer(
            batch["item_embeddings"],
            batch["coarse_category_ids"],
            batch["item_mask"],
        )

    logit = float(scorer_output["compatibility_logit"].item())
    print("Frozen V5 compatibility_logit:", logit)
    print("NOTE: raw uncalibrated logit, NOT a probability or 0–100 score.")
else:
    print("Frozen V5 scorer smoke test skipped because garment count is outside 3–8.")


## Cách đọc kết quả

Tách lỗi thành ba tầng:

1. **Localization failure** — RF-DETR thiếu/sai box → xử lý detector/threshold.
2. **Category failure** — box đúng nhưng Core-7 sai → xem FashionCLIP zero-shot và diagnostic RF-DETR mapping.
3. **Scorer/domain-shift failure** — box + category đúng, scorer vẫn vô lý → investigate crop/domain shift so với Polyvore product-image embeddings.

Một lần chạy thành công chỉ chứng minh wiring/runtime. Trước khi freeze detection vẫn cần nhiều ảnh thật và labeled validation.
